# SENTINEL-GNSS — Kaggle Training Notebook

**Before running:** Settings ▸ Accelerator ▸ **GPU T4 x2** (or P100)  
**Also required:** Settings ▸ **Internet on** (needed to clone from GitHub)

| What lives where | |
|---|---|
| Code + all data CSVs | GitHub (cloned in Step 1) |
| Feature windows (.npz) | Built fresh in `/kaggle/working/sentinel-gnss/` |
| Checkpoints + figures | Google Drive (Step 3) **and** Kaggle Output tab |

### Google Drive persistence (optional but recommended)
Kaggle sessions expire and `/kaggle/working/` is wiped. To keep checkpoints across sessions:
1. Create a **Google Cloud service account** (console.cloud.google.com → IAM → Service Accounts)
2. Enable the **Google Drive API** on that project
3. Download the JSON key and add it as a Kaggle secret named `GDRIVE_CREDENTIALS`
4. Create a folder in your Drive, share it with the service account email, and add the folder ID as `GDRIVE_FOLDER_ID` secret

If secrets are not configured Step 3 falls back to local-only mode — everything still works, outputs are in the Kaggle Output tab.

## Step 1 — Clone repo from GitHub
All code and processed CSVs are pulled directly from GitHub.  
**Internet must be enabled** in Kaggle notebook settings.

In [ ]:
import os; os.chdir('/kaggle/working')

In [ ]:
import os, shutil

GITHUB_REPO = 'https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation.git'
REPO_DIR    = '/kaggle/working/sentinel-gnss'

%cd /kaggle/working

if os.path.exists(f'{REPO_DIR}/.git'):
    print('Repo already cloned — pulling latest ...')
    %cd {REPO_DIR}
    !git pull
else:
    if os.path.exists(REPO_DIR):
        shutil.rmtree(REPO_DIR)
    print('Cloning repo ...')
    !git clone {GITHUB_REPO} {REPO_DIR}
    %cd {REPO_DIR}

print(f'\nWorking directory: {os.getcwd()}')
!git log --oneline -5

csv_path = f'{REPO_DIR}/data/labelled/sentinel_gnss_labelled.csv'
assert os.path.exists(csv_path), f'CSV not found at {csv_path}'
import pandas as pd
df = pd.read_csv(csv_path)
print(f'\nDataset: {len(df):,} rows × {len(df.columns)} columns — ready.')

## Step 2 — Install extra dependencies + verify GPU

In [ ]:
!pip install -q imbalanced-learn xgboost google-api-python-client google-auth-httplib2 google-auth-oauthlib

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {props.total_memory / 1e9:.1f} GB')
    if torch.cuda.device_count() > 1:
        print(f'GPUs     : {torch.cuda.device_count()} (T4 x2)')
else:
    raise RuntimeError(
        'NO GPU — Settings ▸ Accelerator ▸ GPU T4 x2, then re-run from Step 1.')

## Step 3 — Set up output directories + Google Drive (optional)

Creates local output dirs. If Kaggle secrets `GDRIVE_CREDENTIALS` and `GDRIVE_FOLDER_ID`
are present, also authenticates with Google Drive so checkpoints and figures are mirrored
there at the end of each step. Falls back silently to local-only if secrets are missing.

In [ ]:
import os, json, shutil

REPO_DIR = '/kaggle/working/sentinel-gnss'

OUTPUT_DIRS = [
    f'{REPO_DIR}/results/models/checkpoints',
    f'{REPO_DIR}/results/models/checkpoints_lstm_only',
    f'{REPO_DIR}/results/models/checkpoints_transformer_only',
    f'{REPO_DIR}/results/figures',
    f'{REPO_DIR}/results/baselines',
    f'{REPO_DIR}/results/metrics',
]
for d in OUTPUT_DIRS:
    os.makedirs(d, exist_ok=True)
    print(f'  ✓  {d}')

# ── Google Drive setup ────────────────────────────────────────────────────────
DRIVE_ENABLED = False
drive_service = None
GDRIVE_FOLDER_ID = None

try:
    from kaggle_secrets import UserSecretsClient
    secrets = UserSecretsClient()
    creds_json  = secrets.get_secret('GDRIVE_CREDENTIALS')
    GDRIVE_FOLDER_ID = secrets.get_secret('GDRIVE_FOLDER_ID')

    import tempfile
    from google.oauth2.service_account import Credentials
    from googleapiclient.discovery import build

    creds_dict = json.loads(creds_json)
    creds = Credentials.from_service_account_info(
        creds_dict,
        scopes=['https://www.googleapis.com/auth/drive']
    )
    drive_service = build('drive', 'v3', credentials=creds)
    # Test connection
    drive_service.files().get(fileId=GDRIVE_FOLDER_ID).execute()
    DRIVE_ENABLED = True
    print('\n✅ Google Drive connected — results will be mirrored to Drive.')
    print(f'   Folder ID: {GDRIVE_FOLDER_ID}')
except Exception as e:
    print(f'\n⚠️  Google Drive not configured ({type(e).__name__}: {e})')
    print('   Running in local-only mode — outputs available in the Kaggle Output tab.')
    print('   To enable Drive: add GDRIVE_CREDENTIALS and GDRIVE_FOLDER_ID to Kaggle Secrets.')


def upload_to_drive(local_path: str, filename: str = None):
    """Upload a local file to the configured Drive folder. No-op if Drive disabled."""
    if not DRIVE_ENABLED or drive_service is None:
        return
    from googleapiclient.http import MediaFileUpload
    import mimetypes
    fname = filename or os.path.basename(local_path)
    mime  = mimetypes.guess_type(local_path)[0] or 'application/octet-stream'
    # Check if file already exists in folder (update rather than duplicate)
    q = f"name='{fname}' and '{GDRIVE_FOLDER_ID}' in parents and trashed=false"
    existing = drive_service.files().list(q=q, fields='files(id)').execute().get('files', [])
    media = MediaFileUpload(local_path, mimetype=mime, resumable=True)
    if existing:
        drive_service.files().update(fileId=existing[0]['id'], media_body=media).execute()
    else:
        meta = {'name': fname, 'parents': [GDRIVE_FOLDER_ID]}
        drive_service.files().create(body=meta, media_body=media, fields='id').execute()


def sync_folder_to_drive(local_folder: str, exts: tuple = ('.pt', '.json', '.png', '.pdf', '.md', '.txt')):
    """Upload all matching files from a local folder to Drive. No-op if Drive disabled."""
    if not DRIVE_ENABLED:
        return
    uploaded = 0
    for root, _, files in os.walk(local_folder):
        for f in files:
            if any(f.endswith(e) for e in exts):
                upload_to_drive(os.path.join(root, f))
                uploaded += 1
    if uploaded:
        print(f'  → Synced {uploaded} files to Google Drive')


# Report existing checkpoints
ckpt_dir = f'{REPO_DIR}/results/models/checkpoints'
ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
if ckpts:
    print(f'\nExisting checkpoints ({len(ckpts)}): {", ".join(ckpts)}')
else:
    print('\nNo existing checkpoints — will start fresh.')

## Step 4 — Process new datasets (Deep + Harsh) — Run 12 only

> **Standard workflow: Skip this cell.** CSVs are already committed to GitHub.

| Dataset | Expected rows | Expected DEGRADED% |
|---------|---------------|-----------------|
| HK-Deep-Urban-1  (Whampoa, 10 receivers)  | ~14,000 | ~25–35% |
| HK-Harsh-Urban-1 (Mong Kok, 10 receivers) | ~30,000 | ~35–45% |

In [ ]:
%cd /kaggle/working/sentinel-gnss
import os, pandas as pd

deep_csv  = 'data/processed/urbannav/urbannav_deep_features.csv'
harsh_csv = 'data/processed/urbannav/urbannav_harsh_features.csv'

if os.path.exists(deep_csv) and os.path.exists(harsh_csv):
    df_deep  = pd.read_csv(deep_csv)
    df_harsh = pd.read_csv(harsh_csv)
    print(f"Deep  CSV: {len(df_deep):,} rows — labels: {df_deep['label'].value_counts().to_dict()}")
    print(f"Harsh CSV: {len(df_harsh):,} rows — labels: {df_harsh['label'].value_counts().to_dict()}")
    print("\nCSVs from GitHub — proceed to Step 5.")
else:
    deep_dir  = '/kaggle/input/urbannav-deep/urbanNav_Deep'
    harsh_dir = '/kaggle/input/urbannav-harsh/urbanNav_Harsh'
    if os.path.exists(deep_dir):
        !python src/processing/process_all_datasets.py --source urbannav_deep
    else:
        print(f'[SKIP] {deep_dir} not found')
    if os.path.exists(harsh_dir):
        !python src/processing/process_all_datasets.py --source urbannav_harsh
    else:
        print(f'[SKIP] {harsh_dir} not found')
    !python src/processing/process_all_datasets.py --combine

df = pd.read_csv('data/labelled/sentinel_gnss_labelled.csv')
print(f"\nCombined: {len(df):,} rows × {len(df.columns)} columns")

## Step 5 — Build feature windows

Produces two sets of windows:
- `windows/` — SMOTE-balanced (112K train) — used by **all models** (DL + baselines)
- `windows_no_smote/` — natural distribution (62K train) — kept for reference

SMOTE is applied following **Chawla et al. (2002)** and **Johnson & Khoshgoftaar (2019)**.  
Our 37 features are aggregated statistics (mean_cnr, hdop, etc.) — not raw signals —  
so SMOTE interpolation in this space produces valid intermediate signal states.

In [ ]:
%cd /kaggle/working/sentinel-gnss

# SMOTE windows — primary training set for all models
!python -m src.models.feature_prep --force

# No-SMOTE windows — kept for reference / ablation studies
!python -m src.models.feature_prep --no_smote --force

import numpy as np
for tag, wdir in [('SMOTE (primary)', 'windows'), ('no-SMOTE (reference)', 'windows_no_smote')]:
    print(f'\nWindow shapes [{tag}]:')
    for split in ('train', 'val', 'test'):
        d = np.load(f'data/processed/{wdir}/{split}.npz')
        c = int(np.sum(d['y_5s'] == 0))
        w = int(np.sum(d['y_5s'] == 1))
        g = int(np.sum(d['y_5s'] == 2))
        print(f'  {split:5s}  X={d["X"].shape}  CLEAN={c:,}  WARNING={w:,}  DEGRADED={g:,}')

## Step 6 — Train (full Transformer + BiLSTM)

Trains on SMOTE-balanced windows (consistent with paper architecture and expert recommendations).  
Checkpoints saved locally and synced to Drive after training completes.

In [ ]:
import glob, os
REPO_DIR = '/kaggle/working/sentinel-gnss'
ckpt_dir = f'{REPO_DIR}/results/models/checkpoints'

# ── Uncomment to wipe and start fresh ──
# for f in glob.glob(ckpt_dir + '/*.pt'):
#     os.remove(f); print(f'Removed: {f}')

ckpts = sorted(f for f in os.listdir(ckpt_dir) if f.endswith('.pt'))
print(f'Checkpoints: {ckpts if ckpts else "none — starting fresh"}')
print('Add --resume to continue from last checkpoint.')

In [ ]:
%cd /kaggle/working/sentinel-gnss
# ══════════════════════════════════════════════════════════════════
#  SENTINEL-GNSS — Full Transformer+BiLSTM model
#  Training on SMOTE-balanced windows (Chawla et al. 2002)
#  Architecture: TransformerEncoder(2L,8H,d=128) → BiLSTM(2L,h=256)
#  Loss: focal_gamma=1.0, class_weights=[1.0,2.0,5.0], smoothing=0.1
#  Optimiser: AdamW, patience=50, min_epoch_for_best=15, batch=256
# ══════════════════════════════════════════════════════════════════
!python -m src.models.train --batch_size 256

# Sync checkpoints and training history to Drive
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints')

## Step 7 — Evaluate full model
Loads `checkpoint_best.pt`, runs all 14 analyses, saves figures.

In [ ]:
%cd /kaggle/working/sentinel-gnss
!python -m src.models.evaluate \
    --tune_thresholds \
    --temperature_scaling

# Sync figures and metrics to Drive
sync_folder_to_drive(f'{REPO_DIR}/results/figures')

## Step 8 — View all figures inline

In [ ]:
import glob
from IPython.display import Image, display

figs = sorted(glob.glob('/kaggle/working/sentinel-gnss/results/figures/*.png'))
print(f'{len(figs)} figures:')
for f in figs:
    print(f'  {f.split("/")[-1]}')
    display(Image(filename=f, width=950))

## Step 9 — Baselines (Tier 1–3)

All baselines train on the same SMOTE-balanced windows as the DL model.  
RF uses `class_weight='balanced'`; XGBoost uses per-class `sample_weight` — fair comparison.

In [ ]:
%cd /kaggle/working/sentinel-gnss
!python -m src.models.baselines

sync_folder_to_drive(f'{REPO_DIR}/results/baselines')

## Step 10 — Ablations (Tier 4)

LSTM-only and Transformer-only variants trained on the same SMOTE windows.  
All three DL models use identical data, loss, and hyperparameters — only architecture differs.

In [ ]:
%cd /kaggle/working/sentinel-gnss
# LSTM-only ablation
!python -m src.models.train --model_type lstm_only --batch_size 256
!python -m src.models.evaluate \
    --model_type lstm_only \
    --tune_thresholds \
    --temperature_scaling

sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_lstm_only')

In [ ]:
%cd /kaggle/working/sentinel-gnss
# Transformer-only ablation
!python -m src.models.train --model_type transformer_only --batch_size 256
!python -m src.models.evaluate \
    --model_type transformer_only \
    --tune_thresholds \
    --temperature_scaling

sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_transformer_only')

In [ ]:
%cd /kaggle/working/sentinel-gnss
!python -m src.models.baselines --include_ablations

## Step 11 — Generate Run Summary Document

Collects all metrics from JSON files produced by Steps 6–10 and writes a single
human-readable summary document (`RUN_SUMMARY.md`) plus a machine-readable JSON
(`RUN_SUMMARY.json`).  Both are saved to:
- `/kaggle/working/sentinel-gnss/results/` (Kaggle Output tab)
- Google Drive (if configured in Step 3)

In [ ]:
import json, os, glob
from datetime import datetime, timezone

REPO_DIR  = '/kaggle/working/sentinel-gnss'
FIGS_DIR  = f'{REPO_DIR}/results/figures'
BASE_DIR  = f'{REPO_DIR}/results/baselines'
OUT_DIR   = f'{REPO_DIR}/results'

RUN_TS = datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')

# ── Load all metric JSON files ─────────────────────────────────────────────────
def load_json(path):
    try:
        with open(path) as f:
            return json.load(f)
    except Exception:
        return None

metrics_full   = load_json(f'{FIGS_DIR}/metrics_test.json')
metrics_lstm   = load_json(f'{FIGS_DIR}/metrics_test_lstm_only.json')
metrics_trans  = load_json(f'{FIGS_DIR}/metrics_test_transformer_only.json')
baselines_json = load_json(f'{BASE_DIR}/baseline_comparison.json')
thresholds     = load_json(f'{FIGS_DIR}/tuned_thresholds.json')

# Training histories
hist_full  = load_json(f'{REPO_DIR}/results/models/checkpoints/training_history.json')
hist_lstm  = load_json(f'{REPO_DIR}/results/models/checkpoints_lstm_only/training_history.json')
hist_trans = load_json(f'{REPO_DIR}/results/models/checkpoints_transformer_only/training_history.json')

# ── Helper to extract per-horizon table from metrics dict ─────────────────────
def fmt_model_metrics(m, name):
    if not m:
        return f'  {name}: metrics not found\n'
    lines = [f'### {name}']
    horizons = ['5s', '15s', '30s']
    lines.append(f'| Horizon | Accuracy | MacroF1 | WtF1 | κ | MCC |')
    lines.append(f'|---|---|---|---|---|---|')
    for h in horizons:
        hk = f'+{h}'
        if hk not in m:
            continue
        r = m[hk]
        lines.append(
            f"| {hk} | {r.get('accuracy',0):.4f} | {r.get('macro_f1',0):.4f} | "
            f"{r.get('weighted_f1',0):.4f} | {r.get('kappa',0):.4f} | {r.get('mcc',0):.4f} |"
        )
    # Per-class breakdown at +5s
    if '+5s' in m and 'per_class' in m['+5s']:
        lines.append('')
        lines.append('**Per-class @ +5s:**')
        lines.append('| Class | Precision | Recall | F1 | Support |')
        lines.append('|---|---|---|---|---|')
        for cls_name, vals in m['+5s']['per_class'].items():
            lines.append(
                f"| {cls_name} | {vals.get('precision',0):.3f} | "
                f"{vals.get('recall',0):.3f} | {vals.get('f1',0):.3f} | {vals.get('support',0)} |"
            )
    # Bootstrap CIs
    if 'bootstrap_ci' in m:
        lines.append('')
        lines.append('**Bootstrap 95% CIs:**')
        for h, ci in m['bootstrap_ci'].items():
            mf1 = ci.get('macro_f1', [None, None])
            mcc = ci.get('mcc', [None, None])
            lines.append(f'- {h}: MacroF1=[{mf1[0]:.3f}, {mf1[1]:.3f}]  MCC=[{mcc[0]:.3f}, {mcc[1]:.3f}]')
    return '\n'.join(lines) + '\n'

def fmt_baseline_metrics(b):
    if not b:
        return '  Baseline metrics not found\n'
    lines = ['### Classical ML Baselines']
    lines.append('| Method | +5s MacroF1 | +15s MacroF1 | +30s MacroF1 | +5s MCC |')
    lines.append('|---|---|---|---|---|')
    for method, results in b.items():
        if not isinstance(results, dict):
            continue
        f1_5  = results.get('5s',  {}).get('overall', {}).get('macro_f1', 0)
        f1_15 = results.get('15s', {}).get('overall', {}).get('macro_f1', 0)
        f1_30 = results.get('30s', {}).get('overall', {}).get('macro_f1', 0)
        mcc_5 = results.get('5s',  {}).get('overall', {}).get('mcc', 0)
        lines.append(f'| {method} | {f1_5:.4f} | {f1_15:.4f} | {f1_30:.4f} | {mcc_5:.4f} |')
    return '\n'.join(lines) + '\n'

def fmt_training_history(h, name):
    if not h:
        return f'  {name}: history not found\n'
    best_epoch  = h.get('best_epoch', '?')
    best_f1     = h.get('best_val_f1', '?')
    total_epoch = h.get('total_epochs', '?')
    return (f'- **{name}**: best epoch {best_epoch}/{total_epoch}, '
            f'best val MacroF1 = {best_f1}\n')

# ── Build Markdown document ────────────────────────────────────────────────────
md = []
md.append(f'# SENTINEL-GNSS — Run Summary')
md.append(f'**Generated:** {RUN_TS}  ')
md.append(f'**Repo:** https://github.com/Jorshuare/AI-Based-Prediction-for-GNSS-Signal-Degradation  ')
md.append('')

md.append('## Training Summary')
md.append(fmt_training_history(hist_full,  'Transformer + LSTM (full)'))
md.append(fmt_training_history(hist_lstm,  'LSTM-only ablation'))
md.append(fmt_training_history(hist_trans, 'Transformer-only ablation'))

md.append('## Evaluation Results — Test Set')
md.append(fmt_model_metrics(metrics_full,  'Transformer + LSTM (full model)'))
md.append(fmt_model_metrics(metrics_lstm,  'LSTM-only ablation'))
md.append(fmt_model_metrics(metrics_trans, 'Transformer-only ablation'))
md.append(fmt_baseline_metrics(baselines_json))

if thresholds:
    md.append('## Tuned Thresholds (from val set)')
    for h, t in thresholds.items():
        if isinstance(t, dict):
            md.append(f'- **{h}**: WARN={t.get("warn_thresh","?"):.2f}  DEG={t.get("deg_thresh","?"):.2f}')

md.append('')
md.append('## Data & Split Summary')
try:
    import pandas as pd, numpy as np
    d = np.load(f'{REPO_DIR}/data/processed/windows/train.npz')
    md.append(f'- **Train windows (SMOTE):** {d["X"].shape[0]:,}  '
              f'CLEAN={int(np.sum(d["y_5s"]==0)):,}  '
              f'WARNING={int(np.sum(d["y_5s"]==1)):,}  '
              f'DEGRADED={int(np.sum(d["y_5s"]==2)):,}')
    dv = np.load(f'{REPO_DIR}/data/processed/windows/val.npz')
    md.append(f'- **Val windows:** {dv["X"].shape[0]:,}  '
              f'CLEAN={int(np.sum(dv["y_5s"]==0)):,}  '
              f'WARNING={int(np.sum(dv["y_5s"]==1)):,}  '
              f'DEGRADED={int(np.sum(dv["y_5s"]==2)):,}')
    dt = np.load(f'{REPO_DIR}/data/processed/windows/test.npz')
    md.append(f'- **Test windows:** {dt["X"].shape[0]:,}  '
              f'CLEAN={int(np.sum(dt["y_5s"]==0)):,}  '
              f'WARNING={int(np.sum(dt["y_5s"]==1)):,}  '
              f'DEGRADED={int(np.sum(dt["y_5s"]==2)):,}')
except Exception as e:
    md.append(f'  (window stats unavailable: {e})')

md.append('')
md.append('## Figures Generated')
figs = sorted(glob.glob(f'{FIGS_DIR}/*.png'))
for f in figs:
    md.append(f'- {os.path.basename(f)}')

md_text = '\n'.join(md)

# ── Build JSON summary ────────────────────────────────────────────────────────
summary_json = {
    'run_timestamp':    RUN_TS,
    'training_history': {
        'full_model':         hist_full,
        'lstm_only':          hist_lstm,
        'transformer_only':   hist_trans,
    },
    'test_metrics': {
        'full_model':         metrics_full,
        'lstm_only':          metrics_lstm,
        'transformer_only':   metrics_trans,
    },
    'baselines':        baselines_json,
    'tuned_thresholds': thresholds,
}

# ── Save locally ──────────────────────────────────────────────────────────────
md_path   = f'{OUT_DIR}/RUN_SUMMARY.md'
json_path = f'{OUT_DIR}/RUN_SUMMARY.json'

with open(md_path, 'w') as f:
    f.write(md_text)
with open(json_path, 'w') as f:
    json.dump(summary_json, f, indent=2, default=str)

print(f'✅ Saved: {md_path}')
print(f'✅ Saved: {json_path}')
print()
print(md_text)

# ── Upload to Drive ───────────────────────────────────────────────────────────
upload_to_drive(md_path,   'RUN_SUMMARY.md')
upload_to_drive(json_path, 'RUN_SUMMARY.json')
# Also sync all figures and checkpoints
sync_folder_to_drive(f'{REPO_DIR}/results/figures')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_lstm_only')
sync_folder_to_drive(f'{REPO_DIR}/results/models/checkpoints_transformer_only')

if DRIVE_ENABLED:
    print('\n✅ All results synced to Google Drive.')
else:
    print('\n📁 Results saved locally. Download from the Kaggle Output tab.')
    print('   RUN_SUMMARY.md and RUN_SUMMARY.json are the key files to save.')

## Output

```
results/
├── RUN_SUMMARY.md             ← human-readable metrics summary  ← DOWNLOAD THIS
├── RUN_SUMMARY.json           ← machine-readable full metrics   ← DOWNLOAD THIS
├── models/
│   ├── checkpoints/           ← full model .pt files
│   ├── checkpoints_lstm_only/
│   └── checkpoints_transformer_only/
├── figures/                   ← all evaluation plots (.png/.pdf)
└── baselines/                 ← baseline_comparison.json
```

**If Drive is configured:** everything is automatically uploaded after each step.  
**If Drive is not configured:** download `RUN_SUMMARY.md` and `RUN_SUMMARY.json`  
from the Output tab — paste `RUN_SUMMARY.json` directly into the analysis session.